#**Trabalho 3 – Redes Neurais – Adult Income Dataset (Censo)**


---
**Sumário**

1. Descrição da base e variáveis (entendimento dos dados)

2. Estratégias de pré-processamento, recodificação e balanceamento

3. Análise descritiva dos dados

4. Treinamento e ajuste de hiperparâmetros

5. Avaliação e validação do modelo

6. Interpretação e análise dos resultados



#**1. Descrição da base e variáveis (entendimento dos dados)**



O Adult Income Dataset reúne informações demográficas, ocupacionais, educacionais e financeiras provenientes do censo norte-americano de 1994. Após remover registros com valores inválidos (“?”), obteve-se uma base final com cerca de 30 mil observações. As variáveis incluem idade, escolaridade, ocupação, horas trabalhadas por semana, ganhos e perdas de capital, entre outros atributos relevantes para análise de renda. Abaixo, apresenta-se um resumo das variáveis originais utilizadas no estudo.

**Tabela – Variáveis originais do Adult Income Dataset**

| Variável        | Tipo       | Descrição resumida                           |
|-----------------|------------|-----------------------------------------------|
| age             | numérica   | idade em anos                                 |
| workclass       | categórica | categoria de vínculo ocupacional              |
| fnlwgt          | numérica   | peso final do censo                           |
| education       | categórica | nível educacional                             |
| education.num   | numérica   | escolaridade em anos                          |
| marital.status  | categórica | situação conjugal                             |
| occupation      | categórica | ocupação                                      |
| relationship    | categórica | relação familiar                              |
| race            | categórica | raça declarada                                |
| sex             | categórica | sexo                                           |
| capital.gain    | numérica   | ganho de capital                               |
| capital.loss    | numérica   | perda de capital                               |
| hours.per.week  | numérica   | horas trabalhadas por semana                   |
| native.country  | categórica | país de origem                                 |
| income          | alvo       | renda anual (<=50K ou >50K)                    |


#**2. Estratégias de pré-processamento, recodificação e balanceamento**

In [ ]:
# ============================================================
# TRABALHO 3 – REDES NEURAIS (ADULT INCOME)
# Fluxo completo: dados -> recategorização/engenharia -> EDA -> modelos -> comparação
# ============================================================

# 1. Imports e setup
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, roc_curve, auc

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization, Input

# SMOTE
from imblearn.over_sampling import SMOTE

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

# ============================================================
# 2. Leitura da base
# ============================================================
from google.colab import drive
drive.mount('/content/drive')
caminho = "/content/drive/MyDrive/Trabalhos Aprendizado de Máquina 2025.2/Trabalho 3 - Redes Neurais/adult.csv"
df_raw = pd.read_csv(caminho)

In [ ]:
print("===== QUANTIDADE DE DADOS POR CATEGORIA (BASE ORIGINAL) =====\n")

# Selecionar apenas variáveis categóricas (object)
cat_cols = df_raw.select_dtypes(include="object").columns.tolist()

for col in cat_cols:
    print(f"\n--- {col} ---")
    print(df_raw[col].value_counts(dropna=False))
    print("Total de linhas na base original:", df_raw.shape[0])



In [ ]:
# ============================================================
# PIPELINE ÚNICO E LIMPO
# - Tratamento + Feature Engineering (igual ao que elevou a acurácia)
# - One-hot uma única vez
# - Split estratificado
# - SMOTE só no treino
# - Keras com StandardScaler
# - Sklearn MLPClassifier com MinMaxScaler
# - Tabela final consolidada + análises
# ============================================================

import numpy as np
import pandas as pd
import random
import tensorflow as tf

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

from imblearn.over_sampling import SMOTE

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization, Input
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.optimizers import Adam

from sklearn.neural_network import MLPClassifier

# ----------------------------
# 0. Reprodutibilidade
# ----------------------------
SEED = 42
np.random.seed(SEED)
random.seed(SEED)
tf.random.set_seed(SEED)


# -----------------------------------------------
# 1. Base original
# -----------------------------------------------
df = pd.read_csv("/content/drive/MyDrive/Seletivo_IPEA_2025/Trabalho3-RedesNeurais/adult.csv")

# limpeza básica
for col in df.select_dtypes(include="object"):
    df[col] = df[col].astype(str).str.strip()

df = df.replace("?", np.nan)
df = df.dropna().reset_index(drop=True)

# ============================================================
# 2. FEATURE ENGINEERING (EXATAMENTE O QUE MUDOU A ACURÁCIA)
# ============================================================

# Workclass agrupado
df["workclass_pt"] = df["workclass"].replace({
    "Federal-gov":"Publico", "State-gov":"Publico", "Local-gov":"Publico",
    "Private":"Privado", "Self-emp-inc":"Privado","Self-emp-not-inc":"Privado",
    "Without-pay":"Sem-trabalho","Never-worked":"Desempregado"
})

# Education agrupado
map_edu = {
    "Preschool":"Fundamental", "1st-4th":"Fundamental", "5th-6th":"Fundamental",
    "7th-8th":"Fundamental","9th":"Fundamental","10th":"Fundamental","11th":"Fundamental",
    "12th":"Medio","HS-grad":"Medio","Some-college":"Superior","Assoc-voc":"Medio",
    "Assoc-acdm":"Medio","Bachelors":"Superior","Masters":"Pos","Prof-school":"Medio",
    "Doctorate":"Pos"
}
df["education_grp"] = df["education"].map(map_edu)

# Marital agrupado
df["marital_grp"] = df["marital.status"].replace({
    "Married-civ-spouse":"Casado","Married-AF-spouse":"Casado",
    "Married-spouse-absent":"Casado",
    "Never-married":"Solteiro",
    "Divorced":"Separado/Divorciado/Viuvo", "Separated":"Separado/Divorciado/Viuvo","Widowed":"Separado/Divorciado/Viuvo"
})

# Occupation agrupada
df["occupation_grp"] = df["occupation"].replace({
    "Exec-managerial":"Profissionais/Alta Qualificação", "Prof-specialty":"Profissionais/Alta Qualificação","Tech-support":"Profissionais/Alta Qualificação",
    "Sales":"Profissionais/Alta Qualificação",
    "Adm-clerical":"Servico","Craft-repair":"Servico","Farming-fishing":"Servico",
    "Handlers-cleaners":"Servico","Machine-op-inspct":"Servico","Other-service":"Servico",
    "Priv-house-serv":"Servico","Protective-serv":"Servico","Transport-moving":"Servico",
    "Armed-Forces":"Outros"
})

# Relationship agrupada
df["relationship_grp"] = df["relationship"].replace({
    "Husband":"Chefe de família","Wife":"Chefe de família",
    "Own-child":"Dependente",
    "Not-in-family":"Independente","Unmarried":"Independente",
    "Other-relative":"Independente"
})

# Race agrupada
df["race_grp"] = df["race"].replace({
    "White":"Branca","Black":"Negra",
    "Amer-Indian-Eskimo":"Outros","Asian-Pac-Islander":"Outros","Other":"Outros"
})

# País -> continente
continent_map = {
    "United-States":"América Central","Canada":"América Central","Outlying-US(Guam-USVI-etc)":"América Central",
    "England":"Europa","Germany":"Europa","Greece":"Europa","Ireland":"Europa","Italy":"Europa","Poland":"Europa",
    "Portugal":"Europa","Scotland":"Europa","Yugoslavia":"Europa","France":"Europa","Holand-Netherlands":"Europa",
    "Hungary":"Europa",
    "China":"Asia","Japan":"Asia","India":"Asia","Iran":"Asia","Philippines":"Asia","Vietnam":"Asia",
    "Hong":"Asia","Cambodia":"Asia","Taiwan":"Asia","Thailand":"Asia","Laos":"Asia"
}
df["continent"] = df["native.country"].map(continent_map).fillna("Outr")

# Features derivadas (continuidade do seu pipeline)
df["net_capital"] = df["capital.gain"] - df["capital.loss"]
#df["education_hours"] = df["education.num"] * df["hours.per.week"]

# Remoção das colunas originais redundantes
df = df.drop(columns=[
    "workclass","education","marital.status","occupation","relationship","race",
    "native.country","fnlwgt"
])

# target binário
df["income_bin"] = (df["income"] == ">50K").astype(int)
df = df.drop(columns=["income"])

# ----------------------------
# 3. One-hot encoding único
# ----------------------------
df_encoded = pd.get_dummies(df, drop_first=True)

X = df_encoded.drop(columns=["income_bin"])
y = df_encoded["income_bin"]

# ----------------------------
# 4. Split (fixo e estratificado)
# ----------------------------
Xa_tr_raw, Xa_te_raw, ya_tr, ya_te = train_test_split(
    X, y, test_size=0.25, random_state=SEED, stratify=y
)

# ----------------------------
# 5. SMOTE somente no treino
# ----------------------------
sm = SMOTE(random_state=SEED)
Xa_tr_sm_raw, ya_tr_sm = sm.fit_resample(Xa_tr_raw, ya_tr)



#**3. Análise descritiva dos dados**

In [ ]:
# ============================================================
# GRÁFICOS 2×2 – BARRAS EMPILHADAS (VALORES ABSOLUTOS)
# RÓTULO DENTRO DAS BARRAS E TOTAL POR CATEGORIA
# ============================================================

import matplotlib.pyplot as plt
import seaborn as sns

# Base tratada
df_plot = df.copy()
df_plot["income_bin_label"] = df_plot["income_bin"].map({0: "<=50K", 1: ">50K"})

# Renomear variáveis (fora da função)
df_plot = df_plot.rename(columns={
    "workclass_pt": "tipo_trabalho",
    "education_grp": "nivel_educacao",
    "marital_grp": "estado_civil",
    "occupation_grp": "ocupacao",
    "relationship_grp": "relacao_familiar",
    "race_grp": "raca",
})

# Lista final de variáveis categóricas
vars_categoricas = [
    "tipo_trabalho",
    "nivel_educacao",
    "estado_civil",
    "ocupacao",
    "relacao_familiar",
    "raca"
]

# ----------------------------------------------------------
# Função do gráfico — indentação corrigida
# ----------------------------------------------------------
def plot_stacked_counts(ax, data, var):
    tab = pd.crosstab(data[var], data["income_bin_label"])
    categorias = tab.index.tolist()

    bottom_vals = None
    colors = ["#6baed6", "#2171b5"]

    # Criar barras empilhadas
    for i, col in enumerate(tab.columns):
        ax.bar(
            categorias,
            tab[col].values,
            bottom=bottom_vals,
            label=col,
            color=colors[i]
        )

        # Rótulo interno
        for j, val in enumerate(tab[col].values):
            if val > 0:
                ax.text(
                    j,
                    (bottom_vals[j] if bottom_vals is not None else 0) + val/2,
                    str(val),
                    ha="center",
                    va="center",
                    color="white",
                    fontsize=9,
                    fontweight="bold"
                )

        bottom_vals = tab[col].values if bottom_vals is None else bottom_vals + tab[col].values

    # Rótulo externo (apenas o número)
    for j, total in enumerate(bottom_vals):
        ax.text(
            j,
            total + (total * 0.02),
            f"{total}",
            ha="center",
            va="bottom",
            fontsize=9,
            fontweight="bold"
        )

    ax.set_title(var)
    ax.set_ylabel("Contagem")
    ax.set_xticks(range(len(categorias)))
    ax.set_xticklabels(categorias, rotation=45, ha="right")
    ax.legend(title="Renda")


# ----------------------------------------------------------
# Gerar painéis 2×2
# ----------------------------------------------------------
for i in range(0, len(vars_categoricas), 4):
    subset = vars_categoricas[i:i+4]

    fig, axes = plt.subplots(2, 2, figsize=(15, 10))
    axes = axes.flatten()

    for ax, var in zip(axes, subset):
        plot_stacked_counts(ax, df_plot, var)

    for ax in axes[len(subset):]:
        ax.axis("off")

    plt.tight_layout()
    plt.show()


#**4. Treinamento e ajuste de hiperparâmetros**

Keras

In [ ]:
!pip install keras-tuner
import keras_tuner as kt
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization
from tensorflow.keras.optimizers import Adam

def build_model(hp):

    model = Sequential()

    # Número de camadas
    n_layers = hp.Int("n_layers", min_value=1, max_value=4)

    # Primeira camada
    model.add(Dense(
        units=hp.Int("units_0", min_value=32, max_value=512, step=32),
        activation=hp.Choice("activation", ["relu", "tanh"])
    ))

    # Demais camadas
    for i in range(1, n_layers):
        model.add(Dense(
            units=hp.Int(f"units_{i}", min_value=32, max_value=256, step=32),
            activation=hp.Choice("activation_hidden", ["relu", "tanh"])
        ))

        # dropout opcional
        if hp.Boolean(f"dropout_{i}"):
            model.add(Dropout(rate=hp.Float(f"dropout_rate_{i}", 0.1, 0.5, step=0.1)))

    # saída
    model.add(Dense(1, activation="sigmoid"))

    model.compile(
        optimizer=Adam(
            learning_rate=hp.Float("lr", 1e-4, 1e-2, sampling="log")
        ),
        loss="binary_crossentropy",
        metrics=["accuracy"]
    )

    return model
tuner = kt.RandomSearch(
    build_model,
    objective=kt.Objective("val_accuracy", direction="max"),
    max_trials=3,
    executions_per_trial=1,
    overwrite=True,
    directory="keras_tuning",
    project_name="adult_income"
)

tuner.search(
    Xa_tr, ya_tr,
    epochs=40,
    validation_split=0.2,
    batch_size=256,
    verbose=1
)
best_model = tuner.get_best_models(1)[0]
y_prob = best_model.predict(Xa_te).ravel()
y_pred = (y_prob >= 0.5).astype(int)

print("Acurácia:", accuracy_score(ya_te, y_pred))
print("Precisão:", precision_score(ya_te, y_pred))
print("Recall:", recall_score(ya_te, y_pred))
print("F1:", f1_score(ya_te, y_pred))
best_hp = tuner.get_best_hyperparameters(num_trials=1)[0]

print("\n===== MELHORES PARÂMETROS – KERAS TUNER =====")
for p, v in best_hp.values.items():
    print(f"{p}: {v}")


Sckit-learn

In [ ]:
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import RandomizedSearchCV
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import f1_score, precision_score, recall_score, accuracy_score
from imblearn.over_sampling import SMOTE
from scipy.stats import uniform
import numpy as np

# ============================================================
# 1. SMOTE
# ============================================================
sm = SMOTE(random_state=42)
Xtr_sm, ytr_sm = sm.fit_resample(Xa_tr, ya_tr)

# ============================================================
# 2. Normalização
# ============================================================
scaler = MinMaxScaler()
X_train_scaled = scaler.fit_transform(Xtr_sm)
X_test_scaled  = scaler.transform(Xa_te)

# ============================================================
# 3. Modelo base (SEM class_weight)
# ============================================================
mlp_base = MLPClassifier(
    hidden_layer_sizes=(132, 64, 64, 32),
    activation="relu",
    solver="adam",
    learning_rate="adaptive",
    early_stopping=True,
    max_iter=2000,
    random_state=42
)

# ============================================================
# 4. Espaço de busca
# ============================================================
param_dist = {
    "alpha": uniform(1e-6, 1e-3),
    "learning_rate_init": uniform(1e-4, 5e-3),
    "activation": ["relu", "tanh"],
    "batch_size": [64, 128, 256],
    "tol": [1e-4, 1e-5, 1e-6]
}

# ============================================================
# 5. RandomizedSearch
# ============================================================
random_search = RandomizedSearchCV(
    estimator=mlp_base,
    param_distributions=param_dist,
    n_iter=20,
    scoring="f1",
    cv=3,
    random_state=42,
    verbose=2,
    n_jobs=-1
)

random_search.fit(X_train_scaled, ytr_sm)

print("\n===== MELHORES PARÂMETROS =====")
print(random_search.best_params_)

print("\n===== MELHOR F1 MÉDIO =====")
print(random_search.best_score_)

# ============================================================
# 6. Avaliação com threshold tuning
# ============================================================
best_model = random_search.best_estimator_
y_prob = best_model.predict_proba(X_test_scaled)[:, 1]

ths = np.linspace(0.1, 0.9, 81)
best_f1, best_th = -1, 0.5

for th in ths:
    y_pred_th = (y_prob >= th).astype(int)
    f = f1_score(ya_te, y_pred_th)
    if f > best_f1:
        best_f1, best_th = f, th

y_pred_final = (y_prob >= best_th).astype(int)

print("\n===== MELHOR THRESHOLD =====")
print(best_th)

print("\n===== MÉTRICAS FINAIS =====")
print("Acurácia:", accuracy_score(ya_te, y_pred_final))
print("Precisão:", precision_score(ya_te, y_pred_final))
print("Recall:", recall_score(ya_te, y_pred_final))
print("F1:", f1_score(ya_te, y_pred_final))


#**5. Avaliação e validação do modelo**

In [ ]:
# ============================================================
# 6. SCALERS (Keras Standard / Sklearn MinMax)
# ============================================================

# Para Keras
scaler_std = StandardScaler()
Xa_tr = scaler_std.fit_transform(Xa_tr_raw).astype("float32")
Xa_te = scaler_std.transform(Xa_te_raw).astype("float32")

Xa_tr_sm = scaler_std.fit_transform(Xa_tr_sm_raw).astype("float32")

ya_tr_f = ya_tr.astype("float32")
ya_te_f = ya_te.astype("float32")
ya_tr_sm_f = ya_tr_sm.astype("float32")

# Para Sklearn MLPClassifier (MinMax)
scaler_mm = MinMaxScaler()
Xa_tr_mm = scaler_mm.fit_transform(Xa_tr_raw)
Xa_te_mm = scaler_mm.transform(Xa_te_raw)

Xa_tr_sm_mm = scaler_mm.fit_transform(Xa_tr_sm_raw)


# ============================================================
# 7. FUNÇÕES DOS MODELOS
# ============================================================

def build_mlp(input_dim, hidden_layers, dropout=0.0, batch_norm=False, lr=0.001):
    model = Sequential()
    model.add(Input(shape=(input_dim,)))

    for units in hidden_layers:
        model.add(Dense(units, activation="relu"))
        if batch_norm:
            model.add(BatchNormalization())
        if dropout > 0:
            model.add(Dropout(dropout))

    model.add(Dense(1, activation="sigmoid"))

    model.compile(
        optimizer=Adam(learning_rate=lr),
        loss="binary_crossentropy",
        metrics=["accuracy"]
    )
    return model


def run_keras_model(name, cfg, Xtr, Xte, ytr, yte, class_weight=None, threshold=0.5):
    model = build_mlp(
        input_dim=Xtr.shape[1],
        hidden_layers=cfg["layers"],
        dropout=cfg["dropout"],
        batch_norm=cfg["batch_norm"],
        lr=cfg["lr"]
    )

    es = EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True)

    model.fit(
        Xtr, ytr,
        epochs=cfg["epochs"],
        batch_size=cfg["batch_size"],
        validation_split=0.2,
        callbacks=[es],
        class_weight=class_weight,
        verbose=0
    )

    y_prob = model.predict(Xte, verbose=0).ravel()
    y_pred = (y_prob >= threshold).astype(int)

    return {
        "Modelo": name,
        "Arquitetura": str(cfg["layers"]),
        "Dropout": cfg["dropout"],
        "BatchNorm": cfg["batch_norm"],
        "Ativacao": "relu",
        "Solver": "Adam",
        "LR": cfg["lr"],
        "Epochs": cfg["epochs"],
        "Batch_size": cfg["batch_size"],
        "Threshold": threshold,
        "Acuracia": accuracy_score(yte, y_pred),
        "Precisao": precision_score(yte, y_pred),
        "Recall": recall_score(yte, y_pred),
        "F1": f1_score(yte, y_pred),
        "obj": model
    }


def run_sklearn_mlp(name, Xtr, Xte, ytr, yte):
    mlp = MLPClassifier(
        hidden_layer_sizes=(132, 64, 64, 32),
        activation="tanh",
        solver="adam",
        learning_rate_init=0.0010983689107917987,
        alpha=9.16064345328208e-05,
        batch_size=128,
        tol=1e-6,
        max_iter=1000,
        early_stopping=True,
        random_state=SEED
    )

    mlp.fit(Xtr, ytr)
    y_pred = mlp.predict(Xte)

    return {
        "Modelo": name,
        "Arquitetura": "(132, 64, 64, 32)",
        "Dropout": "-",
        "BatchNorm": "-",
        "Ativacao": "tanh",
        "Solver": "adam",
        "LR": 0.0010983689,
        "Epochs": "auto",
        "Batch_size": 128,
        "Threshold": 0.5,
        "Acuracia": accuracy_score(yte, y_pred),
        "Precisao": precision_score(yte, y_pred),
        "Recall": recall_score(yte, y_pred),
        "F1": f1_score(yte, y_pred),
        "obj": mlp
    }


# ============================================================
# 8. CONFIG DOS MODELOS KERAS
# ============================================================

modelos_keras = {
    "MLP Baseline": {
        "layers": (128, 64, 32),
        "dropout": 0.0,
        "batch_norm": False,
        "lr": 0.001,
        "epochs":100,
        "batch_size": 256
    },
    "Deep MLP": {
        "layers": (256, 128, 64, 32),
        "dropout": 0.0,
        "batch_norm": False,
        "lr": 0.001,
        "epochs":100,
        "batch_size": 256
    },
    "MLP + Dropout + BatchNorm": {
        "layers": (128, 64, 32),
        "dropout": 0.3,
        "batch_norm": True,
        "lr": 0.001,
        "epochs":100,
        "batch_size": 256
    }
}


# ============================================================
# 9. EXECUÇÃO COMPLETA — SALVANDO OS MODELOS
# ============================================================

resultados = []
modelos_treinados = {}

for nome, cfg in modelos_keras.items():

    # -------- SEM SMOTE --------
    out_sem = run_keras_model(
        name=f"{nome} (sem SMOTE)",
        cfg=cfg,
        Xtr=Xa_tr, Xte=Xa_te,
        ytr=ya_tr_f, yte=ya_te_f,
        class_weight={0:1, 1:2},
        threshold=0.5
    )
    resultados.append(out_sem)
    modelos_treinados[out_sem["Modelo"]] = out_sem["obj"]

    # -------- COM SMOTE --------
    out_com = run_keras_model(
        name=f"{nome} (com SMOTE)",
        cfg=cfg,
        Xtr=Xa_tr_sm, Xte=Xa_te,
        ytr=ya_tr_sm_f, yte=ya_te_f,
        class_weight=None,
        threshold=0.5
    )
    resultados.append(out_com)
    modelos_treinados[out_com["Modelo"]] = out_com["obj"]


# -------- SKLEARN SEM SMOTE --------
out_sklearn_sem = run_sklearn_mlp(
    "MLPClassifier RandomSearch (sem SMOTE)",
    Xa_tr_mm, Xa_te_mm, ya_tr, ya_te
)
resultados.append(out_sklearn_sem)
modelos_treinados[out_sklearn_sem["Modelo"]] = out_sklearn_sem["obj"]

# -------- SKLEARN COM SMOTE --------
out_sklearn_com = run_sklearn_mlp(
    "MLPClassifier RandomSearch (com SMOTE)",
    Xa_tr_sm_mm, Xa_te_mm, ya_tr_sm, ya_te
)
resultados.append(out_sklearn_com)
modelos_treinados[out_sklearn_com["Modelo"]] = out_sklearn_com["obj"]


# ============================================================
# 10. TABELA FINAL CONSOLIDADA
# ============================================================

df_resultados = pd.DataFrame(resultados)

df_rank_acc = df_resultados.sort_values("Acuracia", ascending=False).reset_index(drop=True)
display(df_rank_acc)

df_rank_f1 = df_resultados.sort_values("F1", ascending=False).reset_index(drop=True)
display(df_rank_f1)


# ============================================================
# 11. ANÁLISES AUTOMÁTICAS
# ============================================================

top_acc = df_rank_acc.iloc[0]
top_f1  = df_rank_f1.iloc[0]

print("\n===== MELHOR POR ACURÁCIA =====")
print(top_acc)

print("\n===== MELHOR POR F1 =====")
print(top_f1)

print("\nExecução concluída.")


In [ ]:
# ============================================
# BLOCO DE ANÁLISE DO MELHOR MODELO JÁ TREINADO
# ============================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import (
    confusion_matrix, ConfusionMatrixDisplay,
    classification_report, accuracy_score,
    precision_score, recall_score, f1_score
)
from sklearn.inspection import permutation_importance

print("\nIdentificando melhor modelo...")
df_rank_acc = df_resultados.sort_values("Acuracia", ascending=False)
melhor_nome = df_rank_acc.iloc[0]["Modelo"]

print("Melhor modelo:", melhor_nome)

# ------------------------------------------------
# 1) Seleção do modelo real já treinado
# ------------------------------------------------
best_model = modelos_treinados[melhor_nome]

# ------------------------------------------------
# 2) Escolha correta do conjunto de teste
# ------------------------------------------------
if "MLPClassifier" in melhor_nome:
    # modelo sklearn → usa MinMaxScaler
    Xte_final = Xa_te_mm
else:
    # modelos Keras → usam StandardScaler
    Xte_final = Xa_te

# ------------------------------------------------
# 3) Predições no teste
# ------------------------------------------------
if "MLPClassifier" in melhor_nome:
    y_pred_best = best_model.predict(Xte_final)
    y_prob_best = best_model.predict_proba(Xte_final)[:, 1]
else:
    y_prob_best = best_model.predict(Xte_final).ravel()
    y_pred_best = (y_prob_best >= 0.5).astype(int)

# ------------------------------------------------
# 4) Matriz de confusão
# ------------------------------------------------
cm = confusion_matrix(ya_te, y_pred_best)
disp = ConfusionMatrixDisplay(confusion_matrix=cm,
                              display_labels=["<=50K", ">50K"])

plt.figure(figsize=(6, 5))
disp.plot(values_format="d", cmap="Blues")
plt.title(f"Matriz de Confusão — {melhor_nome}")
plt.grid(False)
plt.show()

# ------------------------------------------------
# 5) Relatório de classificação
# ------------------------------------------------
print("\nRelatório de Classificação — Melhor Modelo")
print(classification_report(ya_te, y_pred_best, target_names=["<=50K", ">50K"]))

# ------------------------------------------------
# 6) Permutation Importance (apenas sklearn)
# ------------------------------------------------
if "MLPClassifier" in melhor_nome:
    print("\nCalculando Permutation Importance...")

    perm = permutation_importance(
        estimator=best_model,
        X=Xte_final,
        y=ya_te,
        n_repeats=10,
        random_state=SEED,
        scoring="f1"
    )

    sorted_idx = perm.importances_mean.argsort()[::-1]

    df_perm = pd.DataFrame({
        "variavel": X.columns[sorted_idx],
        "importancia_media_F1": perm.importances_mean[sorted_idx],
        "desvio_padrao": perm.importances_std[sorted_idx]
    })

    print("\nTop 15 variáveis por impacto no F1:")
    display(df_perm.head(15))

    plt.figure(figsize=(8, 10))
    plt.barh(
        df_perm["variavel"].head(20)[::-1],
        df_perm["importancia_media_F1"].head(20)[::-1]
    )
    plt.title("Permutation Importance — Top 20 Features")
    plt.xlabel("Queda média no F1 ao permutar")
    plt.tight_layout()
    plt.show()

else:
    print("\nPermutation Importance não disponível para Keras (usa apenas sklearn).")

# ------------------------------------------------
# 7) Trade-off por threshold
# ------------------------------------------------
if "MLPClassifier" in melhor_nome:
    probas = y_prob_best
else:
    probas = y_prob_best

thresholds = np.linspace(0.1, 0.9, 9)
rows = []

for t in thresholds:
    yp = (probas >= t).astype(int)
    rows.append({
        "threshold": t,
        "accuracy": accuracy_score(ya_te, yp),
        "precision": precision_score(ya_te, yp),
        "recall": recall_score(ya_te, yp),
        "f1": f1_score(ya_te, yp)
    })

df_thr = pd.DataFrame(rows)
print("\nAnálise de trade-off por threshold:")
display(df_thr)


#**6. Interpretação e análise dos resultados**

In [ ]:
# ============================================
# BLOCO DE ANÁLISE DO MELHOR MODELO JÁ TREINADO
# ============================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import (
    confusion_matrix, ConfusionMatrixDisplay, classification_report,
    roc_curve, auc, precision_recall_curve, average_precision_score
)
from sklearn.inspection import permutation_importance
from sklearn.calibration import calibration_curve

# ------------------------------------------------
# 1) IDENTIFICAR MELHOR MODELO
# ------------------------------------------------
df_rank_acc = df_resultados.sort_values("Acuracia", ascending=False)

melhor_nome = df_rank_acc.iloc[0]["Modelo"]
print("Melhor modelo:", melhor_nome)

# Recuperar o modelo real salvo no pipeline
best_model = modelos_treinados[melhor_nome]   # <<<<<< CORRETO

# ------------------------------------------------
# 2) PREDIÇÕES NO TESTE
# ------------------------------------------------
if "MLPClassifier" in melhor_nome:
    y_pred = best_model.predict(Xa_te_mm)
    y_prob = best_model.predict_proba(Xa_te_mm)[:, 1]
else:
    y_prob = best_model.predict(Xa_te).ravel()
    y_pred = (y_prob >= 0.5).astype(int)

# ------------------------------------------------
# 3) MATRIZ DE CONFUSÃO
# ------------------------------------------------
cm = confusion_matrix(ya_te, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["<=50K", ">50K"])

plt.figure(figsize=(6, 5))
disp.plot(values_format="d", cmap="Blues")
plt.title("Matriz de Confusão — Melhor Modelo")
plt.grid(False)
plt.show()

# ------------------------------------------------
# 4) RELATÓRIO DE CLASSIFICAÇÃO
# ------------------------------------------------
print("\nRelatório de Classificação — Melhor Modelo")
print(classification_report(ya_te, y_pred, target_names=["<=50K", ">50K"]))

# ------------------------------------------------
# 5) CURVA ROC / AUC
# ------------------------------------------------
fpr, tpr, _ = roc_curve(ya_te, y_prob)
roc_auc = auc(fpr, tpr)

plt.figure(figsize=(6,5))
plt.plot(fpr, tpr, label=f"AUC = {roc_auc:.4f}")
plt.plot([0,1],[0,1],"--")
plt.grid(True)
plt.xlabel("FPR")
plt.ylabel("TPR")
plt.title("Curva ROC — Melhor Modelo")
plt.legend()
plt.show()

# ------------------------------------------------
# 6) PRECISION-RECALL
# ------------------------------------------------
prec, rec, _ = precision_recall_curve(ya_te, y_prob)
ap = average_precision_score(ya_te, y_prob)

plt.figure(figsize=(6,5))
plt.plot(rec, prec, label=f"AP = {ap:.4f}")
plt.grid(True)
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision-Recall Curve")
plt.legend()
plt.show()

# ------------------------------------------------
# 7) CALIBRATION CURVE
# ------------------------------------------------
frac_pos, mean_pred = calibration_curve(ya_te, y_prob, n_bins=10)

plt.figure(figsize=(6,5))
plt.plot(mean_pred, frac_pos, marker="o")
plt.plot([0,1], [0,1], "--")
plt.grid(True)
plt.xlabel("Probabilidade prevista")
plt.ylabel("Fração real de positivos")
plt.title("Calibration Curve")
plt.show()

# ------------------------------------------------
# 8) PERMUTATION IMPORTANCE
# ------------------------------------------------
perm = permutation_importance(
    estimator=best_model,
    X=Xa_te_mm,
    y=ya_te,
    n_repeats=10,
    random_state=SEED,
    scoring="f1"
)

sorted_idx = perm.importances_mean.argsort()[::-1]

df_perm = pd.DataFrame({
    "variavel": X.columns[sorted_idx],
    "importancia_media_F1": perm.importances_mean[sorted_idx],
    "desvio_padrao": perm.importances_std[sorted_idx]
})

print("\nTop variáveis por impacto no F1:")
display(df_perm.head(15))


In [ ]:
# ============================================================
# INTERPRETABILIDADE AVANÇADA DO MELHOR MODELO
# SHAP • LIME • PDP/ICE • OUTLIERS
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import shap
from lime.lime_tabular import LimeTabularExplainer
from sklearn.inspection import PartialDependenceDisplay
from sklearn.ensemble import IsolationForest
from sklearn.metrics import f1_score

# ------------------------------------------------------------
# 0) Identificação do melhor modelo já treinado
# ------------------------------------------------------------
melhor_nome = df_rank_acc.iloc[0]["Modelo"]
print("Melhor modelo:", melhor_nome)

# Recuperar o objeto real salvo previamente
best_model = modelos_treinados[melhor_nome]

# Seleção correta do conjunto de teste
if "MLPClassifier" in melhor_nome:
    X_test = Xa_te_mm
    y_test = ya_te
    predict_function = best_model.predict_proba
else:
    X_test = Xa_te
    y_test = ya_te_f
    predict_function = best_model.predict

# ------------------------------------------------------------
# 1) SHAP – EXPLICABILIDADE GLOBAL E LOCAL
# ------------------------------------------------------------

print("\nGerando explicações SHAP (KernelExplainer)...")

explainer = shap.KernelExplainer(
    model=predict_function,
    data=X_test[:200]
)

shap_values = explainer.shap_values(X_test[:300], nsamples=150)

# Ajuste para modelos binários
if isinstance(shap_values, list):
    shap_classe1 = shap_values[1]
    base_value = explainer.expected_value[1]
else:
    shap_classe1 = shap_values[:, :, 1] if shap_values.ndim == 3 else shap_values
    base_value = explainer.expected_value

# SHAP Summary Plot
shap.summary_plot(
    shap_classe1,
    X_test[:300],
    feature_names=X.columns
)

# SHAP Bar Plot
shap.summary_plot(
    shap_classe1,
    X_test[:300],
    feature_names=X.columns,
    plot_type="bar"
)

# Waterfall (uma única observação)
idx = 10

exp = shap.Explanation(
    values=shap_classe1[idx],
    base_values=base_value,
    data=X_test[idx],
    feature_names=X.columns
)

shap.plots.waterfall(exp, max_display=20)


# ------------------------------------------------------------
# 2) PERMUTATION IMPORTANCE + PDP/ICE
# ------------------------------------------------------------
from sklearn.inspection import permutation_importance

print("\nCalculando Permutation Importance...")

perm = permutation_importance(
    estimator=best_model,
    X=X_test,
    y=y_test,
    scoring="f1",
    n_repeats=10,
    random_state=42
)

top3 = X.columns[perm.importances_mean.argsort()[::-1]][:3]
print("\nTop 3 variáveis para PDP/ICE:", list(top3))

fig, ax = plt.subplots(figsize=(10, 6))
PartialDependenceDisplay.from_estimator(
    best_model,
    X_test,
    features=list(top3),
    feature_names=X.columns,
    kind="both",
    ax=ax
)
plt.tight_layout()
plt.show()


# ------------------------------------------------------------
# 3) LIME – EXPLICAÇÃO LOCAL
# ------------------------------------------------------------
print("\nGerando explicação LIME...")

explainer_lime = LimeTabularExplainer(
    training_data=X_test,
    feature_names=X.columns,
    class_names=["<=50K", ">50K"],
    mode="classification"
)

exp_lime = explainer_lime.explain_instance(
    data_row=X_test[idx],
    predict_fn=predict_function
)

exp_lime.show_in_notebook(show_table=True)


# ------------------------------------------------------------
# 4) OUTLIERS – ISOLATION FOREST
# ------------------------------------------------------------

print("\nExecutando detecção de outliers via IsolationForest...")

iso = IsolationForest(contamination=0.03, random_state=42)
outlier_labels = iso.fit_predict(X_test)

df_outliers = pd.DataFrame({
    "outlier": outlier_labels,
    "y_test": y_test
})

print("\nDistribuição de outliers detectados:")
print(df_outliers["outlier"].value_counts())

print("\nExemplos de índices de outliers:")
print(np.where(outlier_labels == -1)[0][:10])

print("\nInterpretação concluída.")


In [ ]:
import shap
import matplotlib.pyplot as plt
import pandas as pd

# ============================================================
# 1. Mapa de renomeação (nomes mais claros para interpretação)
# ============================================================

rename_map = {
    "occupation_grp_Servico": "Ocupação: Serviços",
    "relationship_grp_Independente": "Relação: Independente",
    "age": "Idade",
    "education.num": "Anos de estudo",
    "marital_grp_Solteiro": "Estado civil: Solteiro",
    "occupation_grp_Profissionais/Alta Qualificação": "Ocupação: Alta Qualificação",
    "relationship_grp_Dependente": "Relação: Dependente",
    "capital.gain": "Ganho de Capital",
    "hours.per.week": "Horas/Semana",
    "capital.loss": "Perda de Capital",
    "marital_grp_Separado/Divorciado/Viuvo": "Estado civil: Separado/Div/Viuvo",
    "net_capital": "Capital Líquido",
    "education_grp_Superior": "Educação: Superior",
    "sex_Male": "Sexo: Masculino",
    "education_grp_Medio": "Educação: Ensino Médio",
    "workclass_pt_Publico": "Setor: Público",
    "continent_Outr": "Continente: Outros",
    "education_grp_Pos": "Educação: Pós-graduação",
    "race_grp_Outros": "Raça: Outras",
    "race_grp_Negra": "Raça: Negra"
}

# garantir que X_plot tenha MESMO número de linhas que shap_classe1
n_amostras = shap_classe1.shape[0]
X_plot = pd.DataFrame(
    X_test[:n_amostras],
    columns=[rename_map.get(col, col) for col in X.columns]
)

# ============================================================
# 2. SHAP SUMMARY PLOT — versão formatada e explicativa
# ============================================================

plt.figure(figsize=(11, 14))

shap.summary_plot(
    shap_classe1,
    X_plot,
    feature_names=X_plot.columns,
    show=False  # para permitir customização
)

plt.title("Importância das Variáveis — SHAP Summary Plot", fontsize=16, pad=20)
plt.xlabel("Impacto no Modelo (Valor SHAP)", fontsize=12)

# ------------------------------------------------------------
# LEGENDA EXPLICATIVA (fixa no rodapé)
# ------------------------------------------------------------

plt.figtext(
    0.02, -0.06,
    "INTERPRETAÇÃO DO GRÁFICO:\n"
    " • Cor vermelha = valores ALTOS da variável\n"
    " • Cor azul = valores BAIXOS da variável\n"
    " • Valor SHAP > 0 → aumenta a probabilidade de renda >50K\n"
    " • Valor SHAP < 0 → reduz a probabilidade de renda >50K\n"
    " • Cada ponto representa um indivíduo\n"
    " • A posição horizontal mostra quanto aquela variável influencia a predição\n",
    ha="left", fontsize=11
)

plt.tight_layout()
plt.show()


## Interpretação técnica e limitações

O notebook aplica redes neurais à classificação de faixas de renda na base Adult Income, comparando implementações com Keras e `MLPClassifier` do scikit-learn, com diferentes configurações e estratégias de treinamento.

**Resultados registrados:** a busca com Keras informa melhor acurácia de validação de aproximadamente 0,8409. A seção de comparação dos modelos identifica `MLPClassifier RandomSearch (sem SMOTE)` como melhor entre as configurações avaliadas segundo o critério implementado. Não se deve interpretar essas saídas como evidência de superioridade universal de uma arquitetura.

**Limitações:** garantir que imputação, codificação e padronização sejam aprendidas somente no treinamento; utilizar o mesmo conjunto de teste e as mesmas métricas para todos os modelos; documentar distribuição das classes, recall e F1 por classe. Para variáveis sociodemográficas, verificar diferença de desempenho entre grupos e evitar conclusões causais a partir de SHAP. As métricas mencionadas foram lidas de saídas armazenadas, sem novo treinamento.